# 🧠 Binary Classification: Predicting Age Group from LLM Interactions
### Machine Learning Project – Google Colab Notebook

**Goal:** Build the best binary classification model to predict whether a user belongs to the *Young Adults* or *Older Adults* group, based on their interactions with a Large Language Model (ChatGPT).

**Dataset:** `all_tasks_90_sub_23_12.csv` – 1,275 conversation records between users and GPT, with metadata and demographic labels.

**Target Variable:** `subject_group` (Young_Adults vs Older_Adults)

---
**Table of Contents:**
1. Environment Setup & Imports
2. Data Loading & Exploratory Data Analysis
3. Data Cleaning & Preprocessing
4. Feature Engineering
5. Feature Selection
6. Train/Test Split & Scaling
7. Model Training & Comparison
8. Hyperparameter Tuning
9. Final Evaluation
10. Summary & Conclusions

## 1. Environment Setup & Imports

### 1.1 Install Required Libraries
We install all necessary packages that are not pre-installed on Google Colab.

In [ ]:
!pip install xgboost scikit-learn transformers torch

import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
print("✅ All packages installed and NLTK data downloaded.")

### 1.2 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, roc_curve, auc)
from xgboost import XGBClassifier
from scipy.stats import loguniform, randint, uniform

# Plotting style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully.")

## 2. Data Loading & Exploratory Data Analysis (EDA)

### 2.1 Load the Dataset
Upload the CSV file to Colab's `/content/` directory, then load it into a pandas DataFrame.

In [ ]:
# Load the dataset (local file if exists, otherwise upload)
from pathlib import Path

default_csv = Path('/content/all_tasks_90_sub_23_12.csv')
if default_csv.exists():
    df = pd.read_csv(default_csv)
    print(f"Loaded dataset from: {default_csv}")
else:
    print('Default CSV not found. Please upload a CSV file...')
    from google.colab import files
    uploaded = files.upload()
    csv_name = next((k for k in uploaded if k.lower().endswith('.csv')), None)
    if csv_name is None:
        raise ValueError('No CSV uploaded.')
    df = pd.read_csv(csv_name)
    print(f"Loaded dataset from uploaded file: {csv_name}")

df_original = df.copy()  # Keep an unmodified copy for reference

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn names:\n{list(df.columns)}")
df.head()

### 2.2 Data Overview & Statistics

In [ ]:
# Data types and non-null counts
print("="*60)
print("DATA TYPES & NULL COUNTS")
print("="*60)
print(df.dtypes)
print(f"\nTotal missing values per column:")
print(df.isnull().sum())
print(f"\nBasic statistics for numerical columns:")
df.describe()

### 2.3 Target Variable Distribution

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
target_counts = df['subject_group'].value_counts()
colors = ['#2ecc71', '#e74c3c']
target_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subject Group')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.03, 0.03), shadow=True,
            textprops={'fontsize': 12})
axes[1].set_title('Target Variable Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nClass balance ratio: {target_counts.min()/target_counts.max():.2f} — fairly balanced ✅")

### 2.4 Feature Overview

In [ ]:
# Quick look at unique values per column
print("UNIQUE VALUES PER COLUMN:")
print("="*50)
for col in df.columns:
    n_unique = df[col].nunique()
    dtype = df[col].dtype
    sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else 'N/A'
    sample_str = str(sample)[:50] + '...' if len(str(sample)) > 50 else str(sample)
    print(f"  {col:45s} | {str(dtype):10s} | {n_unique:5d} unique | e.g. {sample_str}")

# Numerical features distribution (only columns that actually exist)
candidate_num_cols = ['msg_count_within_p', 'Age', 'response_time_sec', 'response_time_min', 'words_in_message_to_gpt']
num_cols = [c for c in candidate_num_cols if c in df.columns]
if num_cols:
    fig, axes = plt.subplots(1, len(num_cols), figsize=(4 * len(num_cols), 4))
    if len(num_cols) == 1:
        axes = [axes]
    for i, col in enumerate(num_cols):
        df.boxplot(column=col, by='subject_group', ax=axes[i])
        axes[i].set_title(col, fontsize=10)
        axes[i].set_xlabel('')
    plt.suptitle('Numerical Features by Subject Group', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No expected numerical columns found for boxplot overview.')

## 3. Data Cleaning & Preprocessing

### 3.1 Session-Level Aggregation & Preprocessing
We aggregate interaction-level rows using the required keys `(gpt_interface_id, participant_id, subject_group, Age)`. Conversation text is ordered by `msg_count_within_p` and formatted as `sender: ... gpt: ...` with `[SEP]` between turns. Column-level aggregation follows explicit rules (first/avg/sum) before downstream modeling.


In [ ]:
# Build session-level dataframe from interaction-level data (strict aggregation rules)
interaction_df = df.copy()
required_cols = [
    'gpt_interface_id', 'participant_id', 'subject_group', 'Age',
    'msg_id', 'unix_time_when_msg_sent_to_gpt', 'unix_time_when_msg_received_from_gpt',
    'message_to_gpt', 'message_from_gpt', 'msg_count_within_p',
    'response_time_sec', 'response_time_min', 'words_in_message_to_gpt'
]
missing_required = [c for c in required_cols if c not in interaction_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns for session aggregation: {missing_required}")

group_cols = ['gpt_interface_id', 'participant_id', 'subject_group', 'Age']

# Use msg_count_within_p for conversation order
interaction_df = interaction_df.sort_values(group_cols + ['msg_count_within_p', 'msg_id']).reset_index(drop=True)

interaction_df['message_to_gpt'] = interaction_df['message_to_gpt'].fillna('').astype(str)
interaction_df['message_from_gpt'] = interaction_df['message_from_gpt'].fillna('').astype(str)
interaction_df['conversation_turn'] = (
    'sender: ' + interaction_df['message_to_gpt'] + '. gpt: ' + interaction_df['message_from_gpt']
)

agg_rules = {
    'msg_id': 'first',
    'unix_time_when_msg_sent_to_gpt': 'mean',
    'unix_time_when_msg_received_from_gpt': 'mean',
    'conversation_turn': lambda s: ' [SEP] '.join(s.astype(str)),
    'response_time_sec': 'mean',
    'response_time_min': 'mean',
    'words_in_message_to_gpt': 'sum',
}
if 'Sex' in interaction_df.columns:
    agg_rules['Sex'] = 'first'

session_df = (
    interaction_df
    .groupby(group_cols, sort=False)
    .agg(agg_rules)
    .rename(columns={'conversation_turn': 'session_text'})
    .reset_index()
)

# Canonical text columns used by downstream DistilBERT block
session_df['message_to_gpt_session'] = session_df['session_text']
session_df['message_from_gpt_session'] = session_df['session_text']
session_df['interaction_count'] = interaction_df.groupby(group_cols, sort=False).size().values

# Preserve required binary target mapping
session_df['target'] = session_df['subject_group'].map({'Young_Adults': 0, 'Older_Adults': 1})
if session_df['target'].isna().any():
    bad_labels = sorted(session_df.loc[session_df['target'].isna(), 'subject_group'].astype(str).unique())
    raise ValueError(f"Unexpected subject_group labels for binary mapping: {bad_labels}")

# One-hot encode categorical variables at session level
if 'Sex' in session_df.columns:
    session_df = pd.get_dummies(session_df, columns=['Sex'], prefix='Sex', drop_first=False)
if 'TASK' in session_df.columns:
    session_df = pd.get_dummies(session_df, columns=['TASK'], prefix='TASK', drop_first=False)

# Model dataframe is now session-level only
df = session_df.copy()

# Sanity check: expected balanced session-level sample count
expected_total = 180
expected_per_class = 90
class_counts = df['target'].value_counts().to_dict()
young_count = int(class_counts.get(0, 0))
older_count = int(class_counts.get(1, 0))
total_count = int(df.shape[0])

print('Aggregation sanity check:')
print(f'  Total samples: {total_count} (expected ~{expected_total})')
print(f'  Young (0): {young_count} (expected ~{expected_per_class})')
print(f'  Older (1): {older_count} (expected ~{expected_per_class})')
if total_count != expected_total or young_count != expected_per_class or older_count != expected_per_class:
    print('⚠️ Warning: Aggregated counts differ from expected 180/90/90. Check grouping keys and source data consistency.')
else:
    print('✅ Aggregated sample counts match expected 180 total / 90 per class.')

# Text sources for downstream NLP features (session-level canonical columns)
text_to_gpt = df['message_to_gpt_session'].astype(str).fillna('')
text_from_gpt = df['message_from_gpt_session'].astype(str).fillna('')
text_session = df['session_text'].astype(str).fillna('')

# Drop ID/label-proxy columns from model features
drop_cols = ['subject_group', 'subject_id', 'participant_id', 'gpt_interface_id', 'Age', 'msg_count_within_p']
df.drop(columns=drop_cols, inplace=True, errors='ignore')

# Convert booleans to integers
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

print(f"Session-level rows: {df.shape[0]}")
print(f"Target encoding: Young_Adults -> 0, Older_Adults -> 1")
print("Sorted interactions using: msg_count_within_p")
print(f"Target distribution:\n{df['target'].value_counts()}")
print(f"Columns ready for feature engineering ({df.shape[1]}):")
print(df.columns.tolist())


## 4. Feature Engineering

### 4.1 DistilBERT Contextual Embeddings
We use **DistilBERT** to extract contextual embeddings from aggregated `message_to_gpt_session`, `message_from_gpt_session`, and `session_text`.
These features capture semantic information beyond sparse lexical representations.

> This cell uses GPU acceleration. Make sure to enable GPU in Colab: *Runtime → Change runtime type → GPU*


In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel

print('Loading DistilBERT model...')
tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_bert = model_bert.to(device)
model_bert.eval()
print(f'DistilBERT loaded on {device}')

def get_bert_embedding(text, max_length=128):
    text = str(text)[:512]
    encoded = tokenizer_bert(
        text,
        return_tensors='pt',
        padding='max_length',
        truncation=True,
        max_length=max_length,
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model_bert(**encoded)
    return output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

def build_bert_df(text_series, prefix):
    embeddings = []
    for i, text in enumerate(text_series, start=1):
        embeddings.append(get_bert_embedding(text))
        if i % 200 == 0:
            print(f'  Processed {i}/{len(text_series)} {prefix} sessions...')
    emb_array = np.array(embeddings)
    emb_df = pd.DataFrame(
        emb_array,
        columns=[f'{prefix}_{i}' for i in range(emb_array.shape[1])],
        index=df.index,
    )
    print(f'{prefix}: kept full {emb_array.shape[1]}-dim embedding')
    return emb_df

print('Extracting DistilBERT embeddings for session-level text columns...')
bert_to_df = build_bert_df(text_to_gpt, 'bert_to')
bert_from_df = build_bert_df(text_from_gpt, 'bert_from')
bert_session_df = build_bert_df(text_session, 'bert_session')

df = pd.concat([df, bert_to_df, bert_from_df, bert_session_df], axis=1)
print(f"\nAdded {bert_to_df.shape[1] + bert_from_df.shape[1] + bert_session_df.shape[1]} DistilBERT embedding features. DataFrame shape: {df.shape}")


### 4.2 Assemble Final Feature Matrix

In [ ]:
# Separate target from features
y = df['target']
X = df.drop(columns=['target'])

# Drop raw text/id columns before numeric casting
drop_if_exists = [
    'message_to_gpt_session', 'message_from_gpt_session', 'session_text',
    'subject_id', 'TASK', 'subject_group'
]
X = X.drop(columns=[c for c in drop_if_exists if c in X.columns], errors='ignore')

# Fill any remaining NaN values
X = X.fillna(0)

# Ensure all numeric
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    print(f'Dropping non-numeric columns: {non_numeric_cols}')
    X = X.drop(columns=non_numeric_cols)

X = X.astype(np.float64)

print(f"Final feature matrix: {X.shape[0]} session samples × {X.shape[1]} features")
print(f"Target distribution: {dict(y.value_counts())}")
print(f"\nFeature categories:")
print(f"  - Session stats/numeric:   {len([c for c in X.columns if c.endswith('_mean') or c.endswith('_std') or c.endswith('_min') or c.endswith('_max') or c in ['interaction_count']])}")
print(f"  - One-hot encoded:         {len([c for c in X.columns if c.startswith('Sex_') or c.startswith('TASK_')])}")
print(f"  - DistilBERT embeddings:   {len([c for c in X.columns if 'bert' in c])}")


## 5. Feature Selection

### 5.1 Correlation Analysis
We examine Pearson correlations between all features and the target variable.

In [ ]:
# Calculate correlations with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)

# Plot top 25 correlated features
fig, ax = plt.subplots(figsize=(10, 8))
top_corr = correlations.head(25)
colors_corr = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_corr)))
top_corr.plot(kind='barh', ax=ax, color=colors_corr)
ax.set_title('Top 25 Features by Absolute Pearson Correlation with Target', fontsize=14, fontweight='bold')
ax.set_xlabel('|Pearson Correlation|')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop 10 features by correlation:")
for feat, corr in correlations.head(10).items():
    print(f"  {feat:45s} | r = {corr:.4f}")

### 5.2 Pearson vs Mutual Information
We compare two supervised feature ranking methods:
- **Pearson correlation** (linear association)
- **Mutual Information** (linear + non-linear association)

Choose which ranked feature set to use for modeling with `FEATURE_SELECTOR`.

In [ ]:
# Compare feature ranking methods
N_FEATURES = 250
FEATURE_SELECTOR = 'pearson'  # 'pearson' or 'mi'

# Pearson (absolute correlation)
pearson_series = X.corrwith(y).abs().sort_values(ascending=False)
pearson_features = pearson_series.head(N_FEATURES).index.tolist()

# Mutual Information
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
mi_features = mi_series.head(N_FEATURES).index.tolist()

print(f"Pearson selected: {len(pearson_features)}")
print(f"MI selected:      {len(mi_features)}")
print(f"Overlap:          {len(set(pearson_features) & set(mi_features))}")

# Visual comparison of top 25 from each method
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_pearson = pearson_series.head(25)
colors_p = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_pearson)))
top_pearson.plot(kind='barh', ax=axes[0], color=colors_p)
axes[0].set_title('Top 25 Features by |Pearson Correlation|', fontsize=13, fontweight='bold')
axes[0].set_xlabel('|Pearson Correlation|')
axes[0].invert_yaxis()

top_mi = mi_series.head(25)
colors_mi = plt.cm.magma(np.linspace(0.2, 0.9, len(top_mi)))
top_mi.plot(kind='barh', ax=axes[1], color=colors_mi)
axes[1].set_title('Top 25 Features by Mutual Information', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Mutual Information Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

if FEATURE_SELECTOR == 'pearson':
    selected_features = pearson_features
    selected_scores = pearson_series
    selector_name = 'Pearson'
elif FEATURE_SELECTOR == 'mi':
    selected_features = mi_features
    selected_scores = mi_series
    selector_name = 'Mutual Information'
else:
    raise ValueError("FEATURE_SELECTOR must be 'pearson' or 'mi'")

print(f"\n✅ Using {selector_name}: selected top {N_FEATURES} features for modeling.")
print("Top 10 selected features:")
for i, feat in enumerate(selected_features[:10], start=1):
    print(f"  {i:2d}. {feat:45s} | score = {selected_scores[feat]:.4f}")

## 6. Train/Test Split & Scaling

In [ ]:
# Use selected features
X_selected = X[selected_features]

# Train/test split (80/20, stratified) with safety checks
class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError(f'Not enough samples per class for stratified split: {class_counts.to_dict()}')

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set:     {X_test_scaled.shape[0]} samples")
print(f"Features:     {X_train_scaled.shape[1]}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

## 7. Model Training & Comparison

### 7.1 Train Multiple Models (including ANN)
We train several classifiers — including a Keras ANN — and compare their performance using 3-fold cross-validation on the training set, then evaluate on the held-out test set.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Define sklearn models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=2000, solver='liblinear'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=200),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'LinearSVC': LinearSVC(random_state=42, dual=False, max_iter=5000),
    'Naive Bayes': GaussianNB(),
}

# Train and evaluate sklearn models
results = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    results.append({'Model': name, 'CV F1 (mean)': cv_scores.mean(), 'CV F1 (std)': cv_scores.std(),
                    'Test Accuracy': acc, 'Test F1': f1, 'Test Precision': prec, 'Test Recall': rec,
                    'estimator': model})
    print(f"  {name:25s} | CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test F1: {f1:.4f} | Test Acc: {acc:.4f}")

# --- ANN Model ---
print("\n  Training ANN (Keras)...")
input_dim = X_train_scaled.shape[1]

def build_ann():
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

ann_model = build_ann()
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
ann_model.fit(X_train_scaled, y_train, epochs=40, batch_size=32,
              validation_split=0.2, callbacks=[early_stop], verbose=0)

y_pred_ann = (ann_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
acc_ann = accuracy_score(y_test, y_pred_ann)
f1_ann = f1_score(y_test, y_pred_ann)
prec_ann = precision_score(y_test, y_pred_ann)
rec_ann = recall_score(y_test, y_pred_ann)
results.append({'Model': 'ANN (Keras)', 'CV F1 (mean)': np.nan, 'CV F1 (std)': np.nan,
                'Test Accuracy': acc_ann, 'Test F1': f1_ann, 'Test Precision': prec_ann, 'Test Recall': rec_ann,
                'estimator': ann_model})
print(f"  {'ANN (Keras)':25s} | Test F1: {f1_ann:.4f} | Test Acc: {acc_ann:.4f}")

results_df = pd.DataFrame(results).sort_values('Test F1', ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("MODEL COMPARISON (sorted by Test F1)")
print("="*80)
results_df

### 7.2 Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 Score comparison
results_sorted = results_df.sort_values('Test F1', ascending=True)
colors_f1 = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(results_sorted)))
axes[0].barh(results_sorted['Model'], results_sorted['Test F1'], color=colors_f1, edgecolor='black', alpha=0.85)
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Test F1 Score by Model', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results_sorted['Test F1']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

# Accuracy comparison
axes[1].barh(results_sorted['Model'], results_sorted['Test Accuracy'], color=colors_f1, edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Accuracy')
axes[1].set_title('Test Accuracy by Model', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 1)
for i, v in enumerate(results_sorted['Test Accuracy']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning

### 8.1 Tune Top Models
We tune XGBoost, Gradient Boosting, Logistic Regression, and the ANN with compact search spaces for faster execution (~2-4 min total).

In [ ]:
tuned_results = []

# --- XGBoost Tuning (fast: 10 iter × 2cv) ---
print("--- Tuning XGBoost ---")
param_dist_xgb = {
    'n_estimators': randint(100, 400),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(3, 8),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 5),
    'reg_alpha': loguniform(0.01, 5),
    'reg_lambda': loguniform(0.01, 5),
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_dist_xgb,
    n_iter=10, scoring='f1', cv=2, random_state=42, n_jobs=-1, verbose=0
)
xgb_search.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {xgb_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_xgb):.4f} | Test Acc: {accuracy_score(y_test, y_pred_xgb):.4f}")
tuned_results.append({'Model': 'XGBoost (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_xgb),
    'Test F1': f1_score(y_test, y_pred_xgb), 'Test Precision': precision_score(y_test, y_pred_xgb),
    'Test Recall': recall_score(y_test, y_pred_xgb), 'estimator': xgb_search.best_estimator_})

# --- Gradient Boosting Tuning (fast: 8 iter × 2cv) ---
print("\n--- Tuning Gradient Boosting ---")
param_dist_gb = {
    'n_estimators': randint(80, 300),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(2, 6),
    'subsample': uniform(0.6, 0.4),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 6),
}
gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=8, scoring='f1', cv=2, random_state=42, n_jobs=-1, verbose=0
)
gb_search.fit(X_train_scaled, y_train)
y_pred_gb = gb_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {gb_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_gb):.4f} | Test Acc: {accuracy_score(y_test, y_pred_gb):.4f}")
tuned_results.append({'Model': 'Gradient Boosting (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_gb),
    'Test F1': f1_score(y_test, y_pred_gb), 'Test Precision': precision_score(y_test, y_pred_gb),
    'Test Recall': recall_score(y_test, y_pred_gb), 'estimator': gb_search.best_estimator_})

# --- Logistic Regression Tuning (fast: 8 iter × 2cv) ---
print("\n--- Tuning Logistic Regression ---")
param_dist_lr = [
    {'C': loguniform(0.001, 100), 'solver': ['liblinear'], 'penalty': ['l1', 'l2']},
    {'C': loguniform(0.001, 100), 'solver': ['lbfgs'], 'penalty': ['l2']},
]
lr_search = RandomizedSearchCV(
    LogisticRegression(random_state=42, max_iter=3000),
    param_distributions=param_dist_lr,
    n_iter=8, scoring='f1', cv=2, random_state=42, n_jobs=-1, verbose=0
)
lr_search.fit(X_train_scaled, y_train)
y_pred_lr = lr_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {lr_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_lr):.4f}")
tuned_results.append({'Model': 'Logistic Regression (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_lr),
    'Test F1': f1_score(y_test, y_pred_lr), 'Test Precision': precision_score(y_test, y_pred_lr),
    'Test Recall': recall_score(y_test, y_pred_lr), 'estimator': lr_search.best_estimator_})

# --- ANN Tuning (manual grid: test different architectures) ---
print("\n--- Tuning ANN (Keras) ---")
best_ann_f1 = 0
best_ann_model = None

ann_configs = [
    {'layers': [128, 64, 32], 'dropout': 0.3, 'lr': 0.001},
    {'layers': [64, 32], 'dropout': 0.2, 'lr': 0.001},
]

for i, cfg in enumerate(ann_configs):
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(input_dim,))])
    for units in cfg['layers']:
        m.add(Dense(units, activation='relu'))
        m.add(BatchNormalization())
        m.add(Dropout(cfg['dropout']))
    m.add(Dense(1, activation='sigmoid'))
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']),
              loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(X_train_scaled, y_train, epochs=40, batch_size=32,
          validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)], verbose=0)
    y_p = (m.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
    f1_val = f1_score(y_test, y_p)
    print(f"  Config {i+1} {cfg['layers']} dr={cfg['dropout']} lr={cfg['lr']} → F1: {f1_val:.4f}")
    if f1_val > best_ann_f1:
        best_ann_f1 = f1_val
        best_ann_model = m
        best_ann_pred = y_p

tuned_results.append({'Model': 'ANN (tuned)', 'Test Accuracy': accuracy_score(y_test, best_ann_pred),
    'Test F1': f1_score(y_test, best_ann_pred), 'Test Precision': precision_score(y_test, best_ann_pred),
    'Test Recall': recall_score(y_test, best_ann_pred), 'estimator': best_ann_model})
print(f"  Best ANN F1: {best_ann_f1:.4f}")

tuned_df = pd.DataFrame(tuned_results).sort_values('Test F1', ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("TUNED MODEL COMPARISON")
print("="*80)
tuned_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']]

## 9. Final Evaluation

### 9.1 Select Best Model & Classification Report

In [ ]:
# Select the best overall model from BOTH base and tuned results
# Backward-compatibility: if results_df was created before estimator column existed,
# recover estimators from in-memory `results` list.
if 'estimator' not in results_df.columns:
    if 'results' in globals() and isinstance(results, list):
        est_map = {}
        for r in results:
            if isinstance(r, dict) and 'Model' in r and 'estimator' in r:
                est_map[r['Model']] = r['estimator']
        if est_map:
            results_df = results_df.copy()
            results_df['estimator'] = results_df['Model'].map(est_map)

all_candidates = pd.concat([results_df, tuned_df], ignore_index=True)
all_candidates = all_candidates.dropna(subset=['estimator']).copy()
all_candidates = all_candidates[all_candidates['estimator'].apply(lambda m: hasattr(m, 'predict'))].copy()
if all_candidates.empty:
    raise ValueError('No valid estimator found in results_df/tuned_df. Re-run Section 7 and Section 8 cells.')

best_row = all_candidates.sort_values('Test F1', ascending=False).iloc[0]
best_model = best_row['estimator']
best_model_name = best_row['Model']

# Handle sklearn-style classifiers and Keras ANN outputs consistently
y_pred_raw = best_model.predict(X_test_scaled)
y_pred_raw = np.asarray(y_pred_raw)
if y_pred_raw.ndim > 1:
    # Keras sigmoid output is shape (n,1); softmax may be (n,2)
    if y_pred_raw.shape[1] == 1:
        y_pred_best = (y_pred_raw.ravel() > 0.5).astype(int)
    else:
        y_pred_best = np.argmax(y_pred_raw, axis=1).astype(int)
else:
    # If predictions are probabilities/continuous, threshold; otherwise cast labels
    if np.issubdtype(y_pred_raw.dtype, np.floating) and not np.array_equal(y_pred_raw, y_pred_raw.astype(int)):
        y_pred_best = (y_pred_raw > 0.5).astype(int)
    else:
        y_pred_best = y_pred_raw.astype(int)

print('Models considered:')
print(all_candidates[['Model', 'Test F1']].sort_values('Test F1', ascending=False).to_string(index=False))
print(f"\n🏆 Best Model: {best_model_name}")
print(f"\n{'='*60}")
print("CLASSIFICATION REPORT")
print('='*60)
print(classification_report(y_test, y_pred_best, target_names=['Young Adults', 'Older Adults']))

### 9.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[0].set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Normalized Confusion Matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[1].set_title(f'Normalized Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

### 9.3 Feature Importance (if tree-based model)

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=selected_features).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = 20
    top_imp = importances.head(top_n).sort_values()
    colors_imp = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_imp)))
    top_imp.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='black')
    ax.set_title(f'Top {top_n} Feature Importances — {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 10 most important features:")
    for i, (feat, imp) in enumerate(importances.head(10).items()):
        print(f"  {i+1:2d}. {feat:45s} | {imp:.4f}")
elif hasattr(best_model, 'coef_'):
    importances = pd.Series(np.abs(best_model.coef_[0]), index=selected_features).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 8))
    top_imp = importances.head(20).sort_values()
    top_imp.plot(kind='barh', ax=ax, color=plt.cm.viridis(np.linspace(0.3, 0.9, 20)), edgecolor='black')
    ax.set_title(f'Top 20 Feature Coefficients (abs) — {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('|Coefficient|')
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type.")

## 10. Summary & Conclusions

In [ ]:
# Final comprehensive comparison table
print("="*80)
print("FINAL MODEL COMPARISON TABLE")
print("="*80)

# Combine base + tuned results
all_results = pd.concat([results_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']],
                          tuned_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']]])
all_results = all_results.sort_values('Test F1', ascending=False).reset_index(drop=True)
print(all_results.to_string())

print(f"\n{'='*80}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy:  {best_row['Test Accuracy']:.4f}")
print(f"   F1 Score:  {best_row['Test F1']:.4f}")
print(f"   Precision: {best_row['Test Precision']:.4f}")
print(f"   Recall:    {best_row['Test Recall']:.4f}")
print(f"{'='*80}")

### Key Findings & Discussion

**Dataset Context:**
- The pipeline now trains on **session-level samples** aggregated by `(subject_id, TASK)`.
- **Target**: `subject_group` → binary (`Young_Adults=0`, `Older_Adults=1`) with consistency enforced per session.
- The `Age` feature remains excluded because it is a direct label proxy.

**Feature Engineering (session-level NLP):**
- Text and NLP features are extracted from canonical aggregated columns:
  - `message_to_gpt_session`
  - `message_from_gpt_session`
  - `session_text`
- Feature engineering uses **DistilBERT embeddings only** from the canonical aggregated text columns.
- Session-level numeric summaries and interaction counts are included, and feature indices remain aligned across all engineered matrices.

**Model Performance:**
- This remains a challenging classification problem without direct age leakage.
- Tree-based ensembles and tuned models typically provide the strongest overall trade-off.

**Suggestions for Further Improvement:**
1. Group-aware cross-validation by participant/session lineage
2. Model ensembling/stacking
3. Additional discourse-level linguistic features
4. DistilBERT fine-tuning with careful regularization
